In [1]:
import os
import json
import numpy as np
from pathlib import Path

In [2]:
def load_split_data(dataset_id, split_name, base_dir):
    """Load expression and network data for a specific split"""
    dataset_dir = os.path.join(base_dir, f'DS{dataset_id}')
    try:
        expression_path = os.path.join(dataset_dir, f'{split_name}_expression.npy')
        network_path = os.path.join(dataset_dir, f'{split_name}_network.npy')
        
        expression_data = np.load(expression_path)
        network_data = np.load(network_path)
        return expression_data, network_data
    except FileNotFoundError as e:
        print(f"Error loading data for DS{dataset_id} {split_name}: {str(e)}")
        return None, None

def load_split_info(dataset_id, base_dir):
    """Load split info JSON file"""
    try:
        info_path = os.path.join(base_dir, f'DS{dataset_id}', 'split_info_0.80.json')
        with open(info_path, 'r') as f:
            return json.load(f)
    except FileNotFoundError as e:
        print(f"Error loading split info for DS{dataset_id}: {str(e)}")
        return None

def compare_arrays(arr1, arr2, name, tolerance=1e-10):
    """Compare two numpy arrays and return detailed comparison results"""
    if arr1 is None or arr2 is None:
        return {
            'equal': False,
            'error': 'One or both arrays are None',
            'shape1': None if arr1 is None else arr1.shape,
            'shape2': None if arr2 is None else arr2.shape
        }
    
    if arr1.shape != arr2.shape:
        return {
            'equal': False,
            'error': 'Shape mismatch',
            'shape1': arr1.shape,
            'shape2': arr2.shape
        }
    
    if not np.allclose(arr1, arr2, rtol=tolerance, atol=tolerance):
        diff = np.abs(arr1 - arr2)
        return {
            'equal': False,
            'error': 'Content mismatch',
            'max_diff': np.max(diff),
            'mean_diff': np.mean(diff),
            'diff_locations': np.where(diff > tolerance)[0].size
        }
    
    return {
        'equal': True,
        'shape': arr1.shape
    }

def compare_split_info(info1, info2, dataset_id):
    """Compare two split info dictionaries"""
    if info1 is None or info2 is None:
        return {
            'equal': False,
            'error': 'One or both split infos are None'
        }
    
    # Compare everything except timestamp
    info1_no_time = {k: v for k, v in info1.items() if k != 'timestamp'}
    info2_no_time = {k: v for k, v in info2.items() if k != 'timestamp'}
    
    return {
        'equal': info1_no_time == info2_no_time,
        'info1': info1_no_time,
        'info2': info2_no_time
    }

def compare_datasets():
    """Compare datasets between data and data_prev folders"""
    current_dir = './data/splits'
    prev_dir = './data_prev/splits'
    splits = ['train', 'valid', 'test']
    
    # Get all dataset IDs from both directories
    current_datasets = {int(d.name[2:]) for d in Path(current_dir).glob('DS*')}
    prev_datasets = {int(d.name[2:]) for d in Path(prev_dir).glob('DS*')}
    all_datasets = sorted(current_datasets.union(prev_datasets))
    
    results = {}
    
    for dataset_id in all_datasets:
        print(f"\nComparing DS{dataset_id}...")
        dataset_results = {'present_in_both': dataset_id in current_datasets and dataset_id in prev_datasets}
        
        if not dataset_results['present_in_both']:
            dataset_results['error'] = f"Dataset only present in {'current' if dataset_id in current_datasets else 'previous'} folder"
            results[dataset_id] = dataset_results
            continue
        
        # Compare split info
        info_current = load_split_info(dataset_id, current_dir)
        info_prev = load_split_info(dataset_id, prev_dir)
        dataset_results['split_info'] = compare_split_info(info_current, info_prev, dataset_id)
        
        # Compare each split
        dataset_results['splits'] = {}
        for split in splits:
            expr_current, net_current = load_split_data(dataset_id, split, current_dir)
            expr_prev, net_prev = load_split_data(dataset_id, split, prev_dir)
            
            split_comparison = {
                'expression': compare_arrays(expr_current, expr_prev, f"{split}_expression"),
                'network': compare_arrays(net_current, net_prev, f"{split}_network")
            }
            dataset_results['splits'][split] = split_comparison
        
        results[dataset_id] = dataset_results
    
    return results

def print_comparison_report(results):
    """Print a formatted report of the comparison results"""
    print("\nDataset Comparison Report")
    print("=" * 80)
    
    for dataset_id, result in results.items():
        print(f"\nDS{dataset_id}:")
        print("-" * 40)
        
        if not result['present_in_both']:
            print(f"ERROR: {result['error']}")
            continue
        
        # Split info comparison
        split_info = result['split_info']
        print("Split Info:", "MATCH" if split_info['equal'] else "MISMATCH")
        if not split_info['equal']:
            print("  Current:", split_info['info1'])
            print("  Previous:", split_info['info2'])
        
        # Split data comparisons
        for split_name, split_data in result['splits'].items():
            print(f"\n{split_name.capitalize()} Split:")
            
            # Expression data
            expr_comp = split_data['expression']
            if expr_comp['equal']:
                print(f"  Expression: MATCH (shape: {expr_comp['shape']})")
            else:
                print(f"  Expression: MISMATCH")
                print(f"    Error: {expr_comp['error']}")
                if 'max_diff' in expr_comp:
                    print(f"    Max difference: {expr_comp['max_diff']}")
                    print(f"    Mean difference: {expr_comp['mean_diff']}")
                    print(f"    Different values: {expr_comp['diff_locations']}")
            
            # Network data
            net_comp = split_data['network']
            if net_comp['equal']:
                print(f"  Network: MATCH (shape: {net_comp['shape']})")
            else:
                print(f"  Network: MISMATCH")
                print(f"    Error: {net_comp['error']}")
                if 'max_diff' in net_comp:
                    print(f"    Max difference: {net_comp['max_diff']}")
                    print(f"    Mean difference: {net_comp['mean_diff']}")
                    print(f"    Different values: {net_comp['diff_locations']}")

In [3]:
results = compare_datasets()
print_comparison_report(results)


Comparing DS1...

Comparing DS2...

Comparing DS3...

Comparing DS1001...

Comparing DS1002...

Comparing DS1003...

Comparing DS1004...

Dataset Comparison Report

DS1:
----------------------------------------
Split Info: MATCH

Train Split:
  Expression: MATCH (shape: (100, 2700))
  Network: MISMATCH
    Error: Content mismatch
    Max difference: 1.0
    Mean difference: 0.0082
    Different values: 82

Valid Split:
  Expression: MATCH (shape: (100, 2700))
  Network: MISMATCH
    Error: Content mismatch
    Max difference: 1.0
    Mean difference: 0.0048
    Different values: 48

Test Split:
  Expression: MATCH (shape: (100, 2700))
  Network: MISMATCH
    Error: Content mismatch
    Max difference: 1.0
    Mean difference: 0.0046
    Different values: 46

DS2:
----------------------------------------
Split Info: MATCH

Train Split:
  Expression: MATCH (shape: (400, 2700))
  Network: MISMATCH
    Error: Content mismatch
    Max difference: 1.0
    Mean difference: 0.0023375
    Diff